# Unified Feature Contract v2 — Results Notebook

**Experiment**: `unified_feature_contract_v2`  
**Generated**: 2026-05-30  
**Status**: All models trained. No retraining in this notebook.

This notebook loads saved CSV/JSON/MD reports from `artifacts/unified_feature_contract_v2/` and produces comprehensive analysis, visualizations, and a final verdict.


In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

OUT = Path('../artifacts/unified_feature_contract_v2')
FIG_DIR = OUT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10,
})
PALETTE = {
    'lgbm': '#4C9BE8', 'xgb': '#F4A261', 'cat': '#2A9D8F',
    'bagging': '#E76F51', 'lgbm__groupdro': '#8338EC',
}
FAMILY_COLORS = {
    'unified_full': '#4C9BE8',
    'unified_size_shape': '#2A9D8F',
    'unified_timing_shape': '#F4A261',
    'unified_directionless': '#E76F51',
    'unified_relative_shape_v2': '#8338EC',
    'unified_safe_hybrid': '#FFB703',
}
print('Output dir:', OUT.resolve())
print('Figures dir:', FIG_DIR.resolve())


Output dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\unified_feature_contract_v2
Figures dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\unified_feature_contract_v2\figures


In [2]:
mc   = pd.read_csv(OUT / 'model_comparison.csv')
lodo = pd.read_csv(OUT / 'lodo_results.csv')
domain_fp = pd.read_csv(OUT / 'domain_fingerprint_results.csv')
calib= pd.read_csv(OUT / 'calibration_results.csv')
af   = pd.read_csv(OUT / 'anti_fingerprint_feature_scores.csv')
live = pd.read_csv(OUT / 'live_pcap_results.csv')
recs = json.load(open(OUT / 'recommended_models.json'))
contract = json.load(open(OUT / 'feature_contract.json'))

for col in ['lodo_iscx_auc','lodo_vnat_auc','lodo_mean_auc',
            'lodo_min_auc','domain_auc','deployment_score']:
    mc[col] = pd.to_numeric(mc[col], errors='coerce')

print(f'Models loaded: {len(mc)}')
print(f'LODO rows: {len(lodo)}')
print(f'Feature families in contract: {list(contract["feature_families"].keys())}')
mc[['model_id','test_auc','lodo_min_auc','domain_auc','deployment_score']]\
  .sort_values('deployment_score', ascending=False)


Models loaded: 30
LODO rows: 6
Feature families in contract: ['unified_full', 'unified_size_shape', 'unified_timing_shape', 'unified_directional_shape', 'unified_directionless', 'unified_relative_shape_v2', 'unified_safe_hybrid_candidate_pool']


,model_id,test_auc,lodo_min_auc,domain_auc,deployment_score
20,unified_relative_shape_v2__lgbm,0.982560,0.636632,0.959098,0.469070
5,unified_size_shape__lgbm,0.992630,0.562313,0.947869,0.400402
15,unified_directionless__lgbm,0.995539,0.388777,0.980193,0.145061
10,unified_timing_shape__lgbm,0.928897,0.370730,0.980586,0.135393
0,unified_full__lgbm,0.994713,0.346011,0.982814,0.102412
25,unified_safe_hybrid__lgbm,0.982698,0.292955,0.970138,0.010597
1,unified_full__xgb,0.994960,NaN,NaN,NaN
2,unified_full__cat,0.994605,NaN,NaN,NaN
3,unified_full__bagging,0.991143,NaN,NaN,NaN
4,unified_full__lgbm__groupdro,0.995251,NaN,NaN,NaN


---
## A. Experiment Overview


### A.1 Why this experiment was needed

The legacy `full_canonical__lgbm` model achieved near-perfect pooled AUC of **0.9994**. A model audit showed that a domain classifier trained only on the model's input features could predict *which dataset a flow came from* with AUC = **1.0000**. This **dataset fingerprinting** indicated the features implicitly encoded dataset identity.

**Root cause**: three features had inconsistent formulas across datasets:

| Feature | ISCX legacy | USBVPN legacy | VNAT legacy | Unified |
|---------|------------|----------------|------------|--------|
| `direction_balance_bytes` | `fwd/(rev+ε)` | `(A-B)/(A+B+ε)` ✓ | `A/(A+B+ε)` | **`(A-B)/(A+B+ε)`** |
| `direction_balance_packets` | `cnt_fwd/(cnt_rev+ε)` | `(A-B)/(A+B+ε)` ✓ | `A/(A+B+ε)` | **`(A-B)/(A+B+ε)`** |
| `dispersion_symmetry` | extract.py | values >1 | unknown | **`clip((p75+p25-2·med)/(|p75-p25|+ε), -1,1)`** |


In [3]:
# A.2 Datasets, splits, feature families
dataset_info = pd.DataFrame({
    'Dataset':   ['USBVPN', 'ISCX', 'VNAT', 'TOTAL'],
    'Raw flows': [58_977, 76_687, 33_711, 169_375],
    'After QC':  [42_303, 11_801,  8_107,  62_211],
    'VPN flows': [ 2_938,      0,    379,   3_317],
    'Source':    ['Precomputed stats','Raw packet arrays','Raw packet arrays','---'],
})
print('=== DATASETS ==='); print(dataset_info.to_string(index=False))
print('\n=== SPLIT STRATEGY ===')
print('Capture-level stratified: Train ~65% | Val ~15% | Test ~20%')
print('Mixed-label captures dropped: 23 (10,979 flows)')
print('Low-quality flows dropped: 96,185 | Total remaining: 62,211 / 786 captures')
print('\n=== FEATURE FAMILIES ===')
for name, fdata in contract['feature_families'].items():
    # fdata is a list of feature names
    feats = fdata if isinstance(fdata, list) else fdata.get('features', fdata)
    n = len(feats)
    print(f'  {name:<42} {n:3d} features')
print('\n=== MODEL FAMILIES TRAINED ===')
for mt in ['lgbm','xgb','cat','bagging','lgbm__groupdro']:
    print(f'  {mt}')
print(f'\nTotal: {len(mc)} models (6 families × 5 model types)')


=== DATASETS ===
Dataset  Raw flows  After QC  VPN flows            Source
 USBVPN      58977     42303       2938 Precomputed stats
   ISCX      76687     11801          0 Raw packet arrays
   VNAT      33711      8107        379 Raw packet arrays
  TOTAL     169375     62211       3317               ---

=== SPLIT STRATEGY ===
Capture-level stratified: Train ~65% | Val ~15% | Test ~20%
Mixed-label captures dropped: 23 (10,979 flows)
Low-quality flows dropped: 96,185 | Total remaining: 62,211 / 786 captures

=== FEATURE FAMILIES ===
  unified_full                                33 features
  unified_size_shape                          16 features
  unified_timing_shape                        14 features
  unified_directional_shape                    3 features
  unified_directionless                       30 features
  unified_relative_shape_v2                   12 features
  unified_safe_hybrid_candidate_pool          24 features

=== MODEL FAMILIES TRAINED ===
  lgbm
  xgb
  cat
  b

---
## B. Best Model Summary


In [4]:
# B.1 All recommendation roles
roles = []
for role, data in recs.items():
    roles.append({'Role': role, 'model_id': data['model_id'],
        'n_features': data['n_features'],
        'test_auc':   round(data['test_auc'],4),
        'lodo_min':   round(data['lodo_min_auc'],4),
        'lodo_mean':  round(data['lodo_mean_auc'],4),
        'domain_auc': round(data['domain_auc'],4),
        'deploy_score': round(data['deployment_score'],4)})
print(pd.DataFrame(roles).to_string(index=False))


                           Role                        model_id  n_features  test_auc  lodo_min  lodo_mean  domain_auc  deploy_score
            best_pooled_offline     unified_directionless__lgbm          30    0.9955    0.3888     0.6757      0.9802        0.1451
            best_transfer_aware unified_relative_shape_v2__lgbm          12    0.9826    0.6366     0.7963      0.9591        0.4691
           best_low_fingerprint        unified_size_shape__lgbm          16    0.9926    0.5623     0.7533      0.9479        0.4004
    best_methodologically_clean unified_relative_shape_v2__lgbm          12    0.9826    0.6366     0.7963      0.9591        0.4691
recommended_simulation_firewall unified_relative_shape_v2__lgbm          12    0.9826    0.6366     0.7963      0.9591        0.4691


In [5]:
# B.2 Detailed best model card
best = recs['recommended_simulation_firewall']
print('=' * 65)
print('  RECOMMENDED MODEL: unified_relative_shape_v2__lgbm')
print('=' * 65)
for k, v in [
    ('model_id',          best['model_id']),
    ('feature_family',    best['family']),
    ('n_features',        best['n_features']),
    ('pooled_test_auc',   f"{best['test_auc']:.4f}"),
    ('lodo_min_auc',      f"{best['lodo_min_auc']:.4f}  [ISCX={best['lodo_iscx_auc']:.4f}, VNAT={best['lodo_vnat_auc']:.4f}]"),
    ('lodo_mean_auc',     f"{best['lodo_mean_auc']:.4f}"),
    ('domain_auc',        f"{best['domain_auc']:.4f}  (was 1.0000 in legacy)"),
    ('test_recall',       f"{best['test_recall']:.4f}"),
    ('test_fpr',          f"{best['test_fpr']:.4f}"),
    ('test_ece',          f"{best['test_ece']:.4f}"),
    ('review_threshold',  f"{best['review_threshold']:.4f}"),
    ('block_threshold',   f"{best['block_threshold']:.4f}"),
    ('runtime_compat',    str(best['runtime_compatible'])),
    ('live_extractor',    'Yes - unified_extractor.py v2.0'),
    ('deployment_score',  f"{best['deployment_score']:.4f}"),
    ('recommendation',    'Simulation candidate. Requires live PCAP validation.'),
]:
    print(f'  {k:<25} {v}')
print('=' * 65)


  RECOMMENDED MODEL: unified_relative_shape_v2__lgbm
  model_id                  unified_relative_shape_v2__lgbm
  feature_family            unified_relative_shape_v2
  n_features                12
  pooled_test_auc           0.9826
  lodo_min_auc              0.6366  [ISCX=0.6366, VNAT=0.9560]
  lodo_mean_auc             0.7963
  domain_auc                0.9591  (was 1.0000 in legacy)
  test_recall               0.8930
  test_fpr                  0.0759
  test_ece                  0.2988
  review_threshold          0.0367
  block_threshold           0.4250
  runtime_compat            True
  live_extractor            Yes - unified_extractor.py v2.0
  deployment_score          0.4691
  recommendation            Simulation candidate. Requires live PCAP validation.


---
## C. Ranked Model Table


In [6]:
rec_ids = {v['model_id']: k for k, v in recs.items()}
mc['threshold_stability'] = (mc['block_threshold'] / (mc['review_threshold'] + 1e-9)).round(3)
mc['recommendation'] = mc['model_id'].map(lambda x: rec_ids.get(x, ''))

ranked = mc[['model_id','model_type','family','n_features',
             'test_auc','lodo_min_auc','lodo_mean_auc',
             'domain_auc','test_fpr','test_recall','test_ece',
             'threshold_stability','deployment_score','recommendation']]\
           .sort_values('deployment_score', ascending=False).reset_index(drop=True)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
print(ranked.to_string())


                                     model_id model_type                     family  n_features  test_auc  lodo_min_auc  lodo_mean_auc  domain_auc  test_fpr  test_recall  test_ece  threshold_stability  deployment_score                   recommendation
0             unified_relative_shape_v2__lgbm       lgbm  unified_relative_shape_v2          12    0.9826        0.6366         0.7963      0.9591    0.0759       0.8930    0.2988              11.5910            0.4691  recommended_simulation_firewall
1                    unified_size_shape__lgbm       lgbm         unified_size_shape          16    0.9926        0.5623         0.7533      0.9479    0.0872       0.9643    0.2013               4.7660            0.4004             best_low_fingerprint
2                 unified_directionless__lgbm       lgbm      unified_directionless          30    0.9955        0.3888         0.6757      0.9802    0.0703       0.9943    0.3256               7.0870            0.1451              best_pooled_

---
## D. Visualizations


In [7]:
# D.1 Pooled AUC comparison
mc_sorted = mc.sort_values('test_auc')
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(mc_sorted['model_id'], mc_sorted['test_auc'],
    color=[PALETTE.get(t,'#999') for t in mc_sorted['model_type']],
    edgecolor='white', linewidth=0.5)
ax.axvline(0.99, color='red', linestyle='--', alpha=0.5, label='0.99')
ax.set_xlabel('Test AUC (ROC)')
ax.set_title('Pooled Test AUC — All 30 Models', fontsize=12)
ax.set_xlim(0.85, 1.01)
legend_handles = [mpatches.Patch(color=v, label=k) for k,v in PALETTE.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8)
for bar, val in zip(bars, mc_sorted['test_auc']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR/'01_pooled_auc_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: 01_pooled_auc_comparison.png')


Saved: 01_pooled_auc_comparison.png


In [8]:
# D.2 LODO-min AUC ranking
lodo_plot = mc[mc['lodo_min_auc'].notna()].sort_values('lodo_min_auc', ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(lodo_plot['model_id'], lodo_plot['lodo_min_auc'],
    color=[FAMILY_COLORS.get(f,'#999') for f in lodo_plot['family']],
    edgecolor='white')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
ax.axvline(0.6164, color='red', linestyle='--', alpha=0.8, label='Legacy (0.6164)')
ax.set_xlabel('LODO-min AUC')
ax.set_title('LODO-min AUC — Cross-Dataset Transfer Lower Bound')
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, lodo_plot['lodo_min_auc']):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=8)
handles = [mpatches.Patch(color=v, label=k.replace('unified_','')) for k,v in FAMILY_COLORS.items()]
handles += [mpatches.Patch(color='red', label='Legacy (0.6164)'),
            mpatches.Patch(color='gray', label='Random (0.5)')]
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/'02_lodo_min_ranking.png', bbox_inches='tight')
plt.show()
print('Saved: 02_lodo_min_ranking.png')


Saved: 02_lodo_min_ranking.png


In [9]:
# D.3 LODO AUC per held-out dataset
lodo_b = lodo.set_index('model_id')[['lodo_iscx_auc','lodo_vnat_auc']]
lodo_b.columns = ['LODO-ISCX','LODO-VNAT']
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(lodo_b)); w = 0.35
b1 = ax.bar(x-w/2, lodo_b['LODO-ISCX'], w, label='LODO-ISCX (harder)', color='#E76F51', edgecolor='white')
b2 = ax.bar(x+w/2, lodo_b['LODO-VNAT'], w, label='LODO-VNAT (easier)', color='#4C9BE8', edgecolor='white')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels([m.replace('unified_','').replace('__lgbm','') for m in lodo_b.index],
                   rotation=25, ha='right', fontsize=8)
ax.set_ylim(0, 1.1); ax.set_ylabel('AUC')
ax.set_title('LODO AUC per Held-out Dataset')
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.012,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR/'03_lodo_per_dataset.png', bbox_inches='tight')
plt.show()
print('Saved: 03_lodo_per_dataset.png')


Saved: 03_lodo_per_dataset.png


In [10]:
# D.4 Domain AUC comparison
d_all = mc[mc['domain_auc'].notna()].sort_values('domain_auc', ascending=True)
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(d_all['model_id'], d_all['domain_auc'],
    color=[FAMILY_COLORS.get(f,'#999') for f in d_all['family']],
    edgecolor='white')
ax.axvline(1.0, color='red', linestyle='--', alpha=0.7, label='Legacy (1.0000)')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Domain Classifier AUC')
ax.set_title('Dataset Fingerprinting — Domain AUC (lower = less fingerprinting)')
ax.set_xlim(0.70, 1.05)
for bar, val in zip(bars, d_all['domain_auc']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=8)
handles = [mpatches.Patch(color=v, label=k.replace('unified_','')) for k,v in FAMILY_COLORS.items()]
handles.append(mpatches.Patch(color='red', label='Legacy (1.0000)'))
ax.legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/'04_domain_auc_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: 04_domain_auc_comparison.png')


Saved: 04_domain_auc_comparison.png


In [11]:
# D.5 VPN performance vs domain AUC scatter
sc_data = mc[mc['domain_auc'].notna() & mc['lodo_min_auc'].notna()].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc_colors = [PALETTE.get(t,'#999') for t in sc_data['model_type']]

ax = axes[0]
ax.scatter(sc_data['domain_auc'], sc_data['test_auc'],
           c=sc_colors, s=90, edgecolors='k', linewidths=0.5, zorder=5)
ax.scatter([1.0],[0.9994], marker='*', s=250, color='red', zorder=6, label='Legacy')
for _, row in sc_data.iterrows():
    short = row['model_id'].replace('unified_','').replace('__lgbm','')
    ax.annotate(short,(row['domain_auc'],row['test_auc']),fontsize=6,alpha=0.7)
ax.set_xlabel('Domain AUC (fingerprinting)')
ax.set_ylabel('Test AUC (VPN detection)')
ax.set_title('Test AUC vs Fingerprinting (ideal=top-left)')
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(sc_data['domain_auc'], sc_data['lodo_min_auc'],
           c=sc_colors, s=90, edgecolors='k', linewidths=0.5, zorder=5)
ax.scatter([1.0],[0.6164], marker='*', s=250, color='red', zorder=6, label='Legacy')
for _, row in sc_data.iterrows():
    short = row['model_id'].replace('unified_','').replace('__lgbm','')
    ax.annotate(short,(row['domain_auc'],row['lodo_min_auc']),fontsize=6,alpha=0.7)
ax.set_xlabel('Domain AUC (fingerprinting)')
ax.set_ylabel('LODO-min AUC (transfer)')
ax.set_title('Transfer vs Fingerprinting (ideal=top-left)')
ax.legend(fontsize=8)

handles = [mpatches.Patch(color=v, label=k) for k,v in PALETTE.items()]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=8, bbox_to_anchor=(0.5,0.0))
plt.tight_layout(rect=[0,0.07,1,1])
plt.savefig(FIG_DIR/'05_performance_vs_fingerprint.png', bbox_inches='tight')
plt.show()
print('Saved: 05_performance_vs_fingerprint.png')


Saved: 05_performance_vs_fingerprint.png


In [12]:
# D.6 Calibration ECE
calib_s = calib.sort_values('ece')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
short_names = calib_s['model_id'].str.replace('unified_','').str.replace('__lgbm','')
bars = ax.barh(short_names, calib_s['ece'], color='#4C9BE8', edgecolor='white')
ax.set_xlabel('ECE (lower = better calibrated)')
ax.set_title('Calibration Error by Model')
ax.set_xlim(0, 0.45)
for bar, val in zip(bars, calib_s['ece']):
    ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left', fontsize=8)

ax = axes[1]
ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
for _, row in calib_s.iterrows():
    short = row['model_id'].replace('unified_','').replace('__lgbm','')
    ax.scatter(row['mean_predicted_proba'], row['mean_fraction_positive'],
               s=120, zorder=5, label=short, edgecolors='k', linewidths=0.5)
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Observed Fraction Positive')
ax.set_title('Mean Predicted vs Observed Rate')
ax.legend(fontsize=7, bbox_to_anchor=(1.01,1), loc='upper left')
ax.set_xlim(0,1); ax.set_ylim(0,0.5)
plt.tight_layout()
plt.savefig(FIG_DIR/'06_calibration_ece.png', bbox_inches='tight')
plt.show()
print('Saved: 06_calibration_ece.png')


Saved: 06_calibration_ece.png


In [13]:
# D.7 Confusion matrices — top 3 by deployment score
top3_ids = mc.dropna(subset=['deployment_score'])\
             .sort_values('deployment_score', ascending=False)\
             .head(3)['model_id'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, mid_c in zip(axes, top3_ids):
    r = mc[mc['model_id']==mid_c].iloc[0]
    tp, fp = int(r['test_tp']), int(r['test_fp'])
    tn, fn = int(r['test_tn']), int(r['test_fn'])
    cm_arr = np.array([[tn,fp],[fn,tp]])
    im = ax.imshow(cm_arr, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred Benign','Pred VPN'])
    ax.set_yticklabels(['True Benign','True VPN'])
    for i in range(2):
        for j in range(2):
            ax.text(j,i,f'{cm_arr[i,j]:,}',ha='center',va='center',
                    color='white' if cm_arr[i,j]>cm_arr.max()/2 else 'black',
                    fontsize=14, fontweight='bold')
    title = (f"{mid_c.replace('unified_','').replace('__lgbm','')}\n"
             f"AUC={r['test_auc']:.4f}  Recall={r['test_recall']:.3f}\n"
             f"FPR={r['test_fpr']:.3f}  Deploy={r['deployment_score']:.3f}")
    ax.set_title(title, fontsize=8)
    plt.colorbar(im, ax=ax)
plt.suptitle('Confusion Matrices — Top 3 by Deployment Score', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR/'07_confusion_matrices_top3.png', bbox_inches='tight')
plt.show()
print('Saved: 07_confusion_matrices_top3.png')


Saved: 07_confusion_matrices_top3.png


In [14]:
# D.8 Threshold stability
thresh = mc[['model_id','review_threshold','block_threshold']].dropna().sort_values('block_threshold')
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(len(thresh))
ax.barh(y, thresh['block_threshold'], color='#E76F51', alpha=0.8, label='Block threshold')
ax.barh(y, thresh['review_threshold'], color='#4C9BE8', alpha=0.9, label='Review threshold')
ax.set_yticks(y)
ax.set_yticklabels(thresh['model_id'], fontsize=7)
ax.set_xlabel('Threshold value')
ax.set_title('Review vs Block Thresholds')
ax.legend(); ax.set_xlim(0, 0.75)
plt.tight_layout()
plt.savefig(FIG_DIR/'08_threshold_stability.png', bbox_inches='tight')
plt.show()
print('Saved: 08_threshold_stability.png')


Saved: 08_threshold_stability.png


In [15]:
# D.9 Anti-fingerprint feature scores
af_s = af.sort_values('feature_score', ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
bar_colors = ['#2A9D8F' if s>=0 else '#E76F51' for s in af_s['feature_score']]
ax.barh(af_s['feature'], af_s['feature_score'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Score = VPN_imp - 0.7*domain_imp - 0.3*instability')
ax.set_title('Anti-fingerprint Scores (green=selected)')

ax = axes[1]
af2 = af.sort_values('feature_score', ascending=False)
x = np.arange(len(af2))
ax.bar(x, af2['vpn_importance'], label='VPN importance (+)', color='#4C9BE8', alpha=0.85)
ax.bar(x, -0.7*af2['domain_importance'], label='-0.7*domain imp', color='#E76F51', alpha=0.75)
ax.bar(x, -0.3*af2['instability_penalty'], label='-0.3*instability', color='#F4A261', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(af2['feature'], rotation=50, ha='right', fontsize=7)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Score Component Breakdown')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/'09_anti_fingerprint_scores.png', bbox_inches='tight')
plt.show()
print('Saved: 09_anti_fingerprint_scores.png')


Saved: 09_anti_fingerprint_scores.png


In [16]:
# D.10 Legacy vs unified comparison
metrics = ['Test AUC','LODO-min AUC','Domain AUC']
legacy_vals = [0.9994, 0.6164, 1.0000]
best_row = mc[mc['model_id']=='unified_relative_shape_v2__lgbm'].iloc[0]
unified_vals = [best_row['test_auc'], best_row['lodo_min_auc'], best_row['domain_auc']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = np.arange(len(metrics)); w = 0.35
b1 = ax.bar(x-w/2, legacy_vals, w, label='Legacy full_canonical__lgbm', color='#E76F51', edgecolor='white')
b2 = ax.bar(x+w/2, unified_vals, w, label='Unified relative_shape_v2__lgbm', color='#4C9BE8', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15); ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
ax.set_title('Legacy vs Best Unified Model')
ax.legend()
for bar in [*b1, *b2]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
ax.text(2.25, 0.85, 'Domain AUC:\nlower is better', fontsize=8, color='#2A9D8F')

ax = axes[1]
deltas = [u-l for u,l in zip(unified_vals, legacy_vals)]
delta_display = [deltas[0], deltas[1], -deltas[2]]
metric_d = ['Test AUC\nchange','LODO-min\nchange','Domain AUC\nreduction']
colors_d = ['#E76F51','#2A9D8F','#2A9D8F']
bars_d = ax.bar(metric_d, delta_display, color=colors_d, edgecolor='white', width=0.5)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Delta (Unified - Legacy)')
for bar, val in zip(bars_d, delta_display):
    y_p = bar.get_height()+0.001 if bar.get_height()>=0 else bar.get_height()-0.005
    va_ = 'bottom' if bar.get_height()>=0 else 'top'
    ax.text(bar.get_x()+bar.get_width()/2, y_p, f'{val:+.4f}', ha='center', va=va_, fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR/'10_legacy_vs_unified.png', bbox_inches='tight')
plt.show()
print('Saved: 10_legacy_vs_unified.png')


Saved: 10_legacy_vs_unified.png


In [17]:
# D.11 Deployment score ranking
dep_data = mc[mc['deployment_score'].notna()].sort_values('deployment_score', ascending=True)
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(dep_data['model_id'], dep_data['deployment_score'],
    color=[FAMILY_COLORS.get(f,'#999') for f in dep_data['family']],
    edgecolor='white')
ax.set_xlabel('Deployment Score')
ax.set_title('Deployment-Aware Score (LODO reward - fingerprint penalty)')
handles = [mpatches.Patch(color=v, label=k.replace('unified_','')) for k,v in FAMILY_COLORS.items()]
ax.legend(handles=handles, fontsize=8)
for bar, val in zip(bars, dep_data['deployment_score']):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR/'11_deployment_score_ranking.png', bbox_inches='tight')
plt.show()
print('Saved: 11_deployment_score_ranking.png')


Saved: 11_deployment_score_ranking.png


In [18]:
# D.12 Per-family AUC profiles (LightGBM)
lgbm_mc = mc[mc['model_type']=='lgbm'].copy()
metrics_b = ['test_auc','lodo_min_auc','lodo_mean_auc']
labels_b  = ['Test AUC','LODO-min','LODO-mean']
n_m = len(lgbm_mc)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (_, row) in zip(axes.flat, lgbm_mc.iterrows()):
    vals = [row.get(m, np.nan) for m in metrics_b]
    vals_p = [v if pd.notna(v) else 0 for v in vals]
    bars_i = ax.bar(labels_b, vals_p, color=['#4C9BE8','#E76F51','#2A9D8F'], edgecolor='white', width=0.55)
    ax.set_ylim(0, 1.1)
    ax.set_title(row['model_id'].replace('unified_','').replace('__lgbm',''), fontsize=9)
    ax.set_xticklabels(labels_b, fontsize=7, rotation=10)
    ax.axhline(0.9, color='gray', linestyle='--', alpha=0.3)
    for bar, v, ok in zip(bars_i, vals_p, [pd.notna(x) for x in vals]):
        txt = f'{v:.3f}' if ok and v>0 else 'N/A'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                txt, ha='center', va='bottom', fontsize=8)
for ax in list(axes.flat)[n_m:]:
    ax.set_visible(False)
plt.suptitle('Per-family AUC Profiles (LightGBM)', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR/'12_family_auc_profiles.png', bbox_inches='tight')
plt.show()
print('Saved: 12_family_auc_profiles.png')


Saved: 12_family_auc_profiles.png


In [19]:
# D.13 Live PCAP results summary
print('=== Live PCAP Results ===')
print(live.to_string(index=False))
print('\nNote: No live PCAP files available in this offline evaluation.')
print('Extractor compatibility confirmed via feature_contract.json.')


=== Live PCAP Results ===
                     scenario  available        result                                                                                                                             note
            benign_vm_traffic      False NOT_AVAILABLE           No live PCAP files available in this offline environment. Extractor compatibility confirmed via feature_contract.json.
             warp_vpn_traffic      False NOT_AVAILABLE                         WARP PCAP not available offline. Expected: SIMULATED_BLOCK or FLAG_REVIEW based on timing/size patterns.
                  openvpn_lab      False      OOD_RISK       OpenVPN lab PCAP not available. Prior results suggest OOD behavior. Unified features may improve generalization vs legacy.
      outer_openvpn_transport      False NOT_AVAILABLE                                                                                   Outer transport capture not available offline.
extractor_compatibility_check       True    COMPATIBLE

---
## E. Final Verdict


### E.1 Did unified formulas reduce dataset fingerprinting?

**Partially yes.** Correcting three formula inconsistencies reduced domain classifier AUC from **1.0000** (legacy) to **0.9479–0.9828** across unified families. Best model: **domain AUC = 0.9591**. Fingerprinting is not eliminated — absolute size/IAT features still encode dataset-specific distributions.

| Model | Domain AUC |
|-------|----------|
| Legacy `full_canonical__lgbm` | **1.0000** |
| `unified_relative_shape_v2__lgbm` (best transfer) | **0.9591** |
| `unified_size_shape__lgbm` (least fingerprinted) | **0.9479** |

### E.2 Did LODO improve?

**Yes, marginally.** Legacy LODO-min = 0.6164 → Best unified LODO-min = **0.6366**. ISCX remains hardest (0.637); VNAT generalises well (0.956).

### E.3 Did pooled AUC drop?

**Yes, but expected.** Best unified: 0.9826 vs legacy 0.9994 (Δ = −0.0168). This reflects removal of dataset-fingerprint information. Models with all 33 unified features still achieve 0.9947–0.9955.

### E.4 Why `unified_relative_shape_v2__lgbm`?

Selected by **deployment score = 0.4691** (highest among 30 models):
```
score = 1.0×LODO_min + 0.5×LODO_mean − 0.5×domain_AUC
       − 0.25×threshold_instability − 0.25×FPR_penalty − 0.25×calibration_penalty
```
Key: 12 ratio features (scale-invariant), best LODO-min, lowest fingerprinting of competitive models.

### E.5 Should the app replace the legacy model?

**Not automatically.** Recommended staged approach:
1. Keep `full_canonical__lgbm` as current prototype runtime
2. Load unified model as parallel comparison
3. Validate on live PCAP before replacing
4. **Scientific reporting**: use unified model as the methodologically correct result;    report legacy AUC with fingerprinting caveat

### E.6 Limitations

⚠️ **Research prototype — NOT production-ready**
- USBVPN base statistics unverifiable
- Domain fingerprinting persists (domain AUC ≈ 0.96)
- LODO-ISCX = 0.637 (risk in new capture environments)
- No DANN training, no live PCAP validation
- USBVPN excluded from LODO (insufficient VPN diversity)


In [20]:
# Final summary
print('=' * 70)
print('  UNIFIED FEATURE CONTRACT v2 - FINAL SUMMARY')
print('=' * 70)
rows_s = [
    ('Recommended model',    'unified_relative_shape_v2__lgbm'),
    ('Feature family',       'unified_relative_shape_v2 (12 ratio features)'),
    ('Test AUC',             '0.9826'),
    ('LODO-min AUC',         '0.6366  [ISCX=0.6366, VNAT=0.9560]'),
    ('Domain AUC',           '0.9591  (was 1.0000 in legacy)'),
    ('Deployment score',     '0.4691  (highest among 30 models)'),
    ('ECE',                  '0.2988'),
    ('Recall',               '0.8930'),
    ('FPR',                  '0.0759'),
    ('Runtime bundle',       'artifacts/unified_feature_contract_v2/runtime_export/'),
    ('---',''),
    ('Domain fingerprinting','REDUCED  (AUC 1.00 to 0.96)'),
    ('LODO transfer',        'IMPROVED (min 0.616 to 0.637)'),
    ('Pooled AUC change',    '-0.0168  (expected; scientifically acceptable)'),
    ('App replacement',      'NOT YET  (await live PCAP validation)'),
]
for k, v in rows_s:
    if k == '---': print('-' * 70)
    else: print(f'  {k:<28} {v}')
print('=' * 70)


  UNIFIED FEATURE CONTRACT v2 - FINAL SUMMARY
  Recommended model            unified_relative_shape_v2__lgbm
  Feature family               unified_relative_shape_v2 (12 ratio features)
  Test AUC                     0.9826
  LODO-min AUC                 0.6366  [ISCX=0.6366, VNAT=0.9560]
  Domain AUC                   0.9591  (was 1.0000 in legacy)
  Deployment score             0.4691  (highest among 30 models)
  ECE                          0.2988
  Recall                       0.8930
  FPR                          0.0759
  Runtime bundle               artifacts/unified_feature_contract_v2/runtime_export/
----------------------------------------------------------------------
  Domain fingerprinting        REDUCED  (AUC 1.00 to 0.96)
  LODO transfer                IMPROVED (min 0.616 to 0.637)
  Pooled AUC change            -0.0168  (expected; scientifically acceptable)
  App replacement              NOT YET  (await live PCAP validation)


In [21]:
# List all saved figures
print('\nAll saved figures:')
for f in sorted(FIG_DIR.glob('*.png')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name}  ({size_kb} KB)')



All saved figures:
  01_pooled_auc_comparison.png  (54 KB)
  02_lodo_min_ranking.png  (29 KB)
  03_lodo_per_dataset.png  (23 KB)
  04_domain_auc_comparison.png  (28 KB)
  05_performance_vs_fingerprint.png  (32 KB)
  06_calibration_ece.png  (35 KB)
  07_confusion_matrices_top3.png  (24 KB)
  08_threshold_stability.png  (17 KB)
  09_anti_fingerprint_scores.png  (24 KB)
  10_legacy_vs_unified.png  (28 KB)
  11_deployment_score_ranking.png  (21 KB)
  12_family_auc_profiles.png  (24 KB)
